# Page 6: what happens when a model runs out of tokens

A model reasons by writing its chain of thought one token at a time. This notebook gives three
Qwen2.5 models a hard limit on how many tokens they may write, from 128 up to 4,096, and measures
how many of 100 hard maths problems each one still solves. When a model reaches its limit it simply
stops, exactly as it would in use.

The token budget is the only thing that changes. Every run uses the same 100 problems (MATH-500,
levels 4 and 5, the same set as page 5), the same prompt, and greedy decoding, so a model given the
same budget always writes the same chain.

**What it writes**, in `out/page_06/`:

| File | What it holds |
|---|---|
| `traces.jsonl` | every chain the models wrote, token by token |
| `results.json` | every number this notebook computes |
| `manifest.json` | how, where and with what the run was made |

`page_06_site_data.py` turns these into the `derived.json` the website draws. Shared helpers (model
loading, generation from token ids, the resumable log) live in `harness.py`. Both ship beside this
notebook.

## 1. How to run it

| Mode | What it does | Hardware | Time |
|---|---|---|---|
| `REPLAY` (default) | Rebuilds every number and figure from the published chains. Nothing is generated. | any laptop | under a minute |
| `QUICK` | Generates a small run: Qwen2.5-3B, 30 problems, 3 budgets. | GPU, ~6 GB | ~20 minutes |
| `SMALL` | Generates the budget sweep for the 1.5B and 3B on 10 problems, 6 budgets, into `out/page_06_small/`. | GPU, ~8 GB | ~15 minutes |
| `FULL` | Generates the published run: 3 models, 100 problems, 6 budgets. | GPU, 16 GB+ | 3 to 5 hours |

Choose the mode in the next cell, or from a shell:

```bash
PAGE6_RUN_MODE=FULL jupyter nbconvert --to notebook --execute page_06_thinking_longer.ipynb
```

Generation is resumable: each chain is appended to `traces.jsonl` as soon as it is written, so an
interrupted run carries on from where it stopped. Install the dependencies with
`pip install -r requirements.txt`.

In [ ]:
import os

RUN_MODE = os.environ.get("PAGE6_RUN_MODE", "REPLAY")      # "REPLAY", "QUICK", "SMALL" or "FULL"

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"        # the model figures 1 and 3 are drawn from
ALL_MODELS = [                              # (model, precision)
    (PRIMARY, "bf16"),
    ("Qwen/Qwen2.5-1.5B-Instruct", "bf16"),
    ("Qwen/Qwen2.5-7B-Instruct", "8bit"),   # 8-bit so that it fits on a 16 GB card
]
MODES = {
    "REPLAY": dict(models=ALL_MODELS, n_items=100, budgets=[128, 256, 512, 1024, 2048, 4096], waits=[0, 1, 2, 3]),
    "QUICK":  dict(models=ALL_MODELS[:1], n_items=30, budgets=[128, 512, 2048], waits=[0, 1, 2]),
    "FULL":   dict(models=ALL_MODELS, n_items=100, budgets=[128, 256, 512, 1024, 2048, 4096], waits=[0, 1, 2, 3]),
    # A small run on the two smaller models: the budget sweep only, in its own folder.
    "SMALL":  dict(models=ALL_MODELS[:2], n_items=10, budgets=[128, 256, 512, 1024, 2048, 4096], waits=[0, 1, 2, 3],
                   wait_arm=False, out_dir="out/page_06_small"),
}
CFG = MODES[RUN_MODE]
MODELS, N_ITEMS = CFG["models"], CFG["n_items"]
BUDGETS = sorted(CFG["budgets"])
MAX_BUDGET = BUDGETS[-1]
SEED = 123                  # chooses which problems are used; decoding itself is greedy

# Section 12: the opposite of a cap, telling the model to keep going.
RUN_WAIT_ARM = CFG.get("wait_arm", True)
WAIT_WORD = "Wait"
WAITS = sorted(CFG["waits"])     # how many times it is told to keep going
WAIT_ALLOWANCE = 512             # tokens each "Wait" buys

GENERATES = RUN_MODE != "REPLAY"
assert PRIMARY in dict(MODELS), "the primary model must be one of the models in the run"
print(f"mode {RUN_MODE}: {N_ITEMS} problems, budgets {BUDGETS}")
for model_id, quant in MODELS:
    print(f"   {model_id.split('/')[-1]:<24}{quant}")

In [ ]:
import json
import random
import textwrap
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap

import harness
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    extract_final_candidate,
    get_last_boxed,
    grade_answer,
    load_math500_test,
    render_prompt,
)

OUT_DIR, DATA_DIR = Path(CFG.get("out_dir", "out/page_06")), Path("data")
for folder in (OUT_DIR, DATA_DIR):
    folder.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
RESULTS_PATH = OUT_DIR / "results.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean = harness.mean
print("torch", torch.__version__, "| harness", harness.__version__)

## 2. The problems

100 problems from MATH-500 at levels 4 and 5, the two hardest levels. They are drawn with a fixed
seed and the list is hashed, so "the same problems as page 5" can be checked instead of taken on
trust.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
pool = sorted((r for r in math500 if int(r["level"]) in (4, 5)), key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(pool)
items = pool[:N_ITEMS]

ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)
ORDER = {r["unique_id"]: i for i, r in enumerate(items)}     # every list below keeps this order

levels = Counter(int(r["level"]) for r in items)
print(f"{len(items)} of the {len(pool)} level 4 and 5 problems: {levels[4]} at level 4, "
      f"{levels[5]} at level 5   (item set {ITEM_SET_HASH[:12]})")

## 3. Grading

Chapter 3's grader from *Reasoning from Scratch*, unchanged
([`ch03.py`](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py): `extract_final_candidate`, `grade_answer`). `grade_chain` adds one rule
for chains the budget cut off.

| The chain | Its answer | Correct when |
|---|---|---|
| finished inside its budget | its last `\boxed{}`, or its last number if it boxed nothing | the answer equals the reference; equivalent forms match (`\frac{1}{2}` and `0.5`) |
| cut off by its budget | a `\boxed{}` written before the cut, nothing else | the same; no boxed answer is wrong |

In [ ]:
def grade_chain(text, cut_off, reference):
    """Returns (answer, correct). A chain cut off by its budget only counts if it already
    wrote a \boxed{} answer; a finished chain is graded the usual way."""
    if cut_off:
        answer = (get_last_boxed(text) or "").strip().strip("$ ")
    else:
        answer = extract_final_candidate(text)
    return answer, bool(answer) and bool(grade_answer(answer, reference))


harness.check_grader([
    # (what the model wrote, reference answer, should it count as correct?)
    (r"The answer is \boxed{9}.", "9", True),
    (r"27", "9", False),
    (r"\boxed{\frac{1}{2}}", "0.5", True),
    (r"\boxed{1,024}", "1024", True),
    (r"\boxed{\left( 3, \frac{\pi}{2} \right)}", r"\left( 3, \frac{\pi}{2} \right)", True),
], extract_final_candidate, grade_answer)

# Cut off with no boxed answer: wrong, even though the last-number fallback would find "12".
assert grade_chain("so 3 x 4 = 12, and then", cut_off=True, reference="12") == ("", False)
assert grade_chain(r"so the answer is \boxed{12}. Let me also", cut_off=True, reference="12")[1]

## 4. The models

| Model | Precision |
|---|---|
| Qwen2.5-1.5B-Instruct | bf16 |
| Qwen2.5-3B-Instruct (primary: figures 1 and 3) | bf16 |
| Qwen2.5-7B-Instruct | 8-bit, to fit a 16 GB card |

All three share one tokenizer, so a budget of N tokens means the same thing for each. `REPLAY` loads
none of them and reads the published chains.

In [ ]:
results = {}       # results[model][budget]: one record per problem, in problem order
wait_records = {}  # section 12's records, by model
model_meta = {}    # what was loaded, for the manifest


if RUN_MODE == "REPLAY":
    assert TRACES_PATH.exists() and MANIFEST_PATH.exists(), (
        f"REPLAY reads the published run from {OUT_DIR}/traces.jsonl and manifest.json, "
        f"which come with the notebook download.")
    RUN_MANIFEST = json.loads(MANIFEST_PATH.read_text())
    RUN_ID = RUN_MANIFEST["run_id"]
    assert RUN_MANIFEST["items"]["item_set_sha256"] == ITEM_SET_HASH, \
        "the published run was made on a different set of problems"

    with TRACES_PATH.open(encoding="utf-8") as f:
        for r in (json.loads(line) for line in f if line.strip()):
            if r["run_id"] != RUN_ID or r["problem_id"] not in ORDER:
                continue
            if r.get("arm") == "wait":
                wait_records.setdefault(r["model_id"], []).append(r)
            elif r["budget"] in BUDGETS:
                results.setdefault(r["model_id"], {}).setdefault(r["budget"], []).append(r)
    model_meta = {m["id"]: m for m in RUN_MANIFEST["models"]}
    print(f"replaying {RUN_ID}; no model loaded")
else:
    device = get_device()
    DTYPE = harness.pick_dtype(device)
    GPU_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else device.type
    harness.assert_same_gpu(MANIFEST_PATH, GPU_NAME)
    RUN_ID = (f"{RUN_MODE.lower()}-math500_l45-n{len(items)}-{ITEM_SET_HASH[:8]}"
              f"-cap{MAX_BUDGET}-{str(DTYPE).rsplit('.', 1)[-1]}-{device.type}-stop")
    STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("model_id", "budget", "problem_id"))
    STORE.report()

    def load(model_id, quant):
        got = harness.load_model(model_id, device=device, dtype=DTYPE,
                                 load_in_8bit=(quant == "8bit"), warmup_tokens=16)
        mdl, tok = got["model"], got["tokenizer"]
        model_meta[model_id] = dict(
            id=model_id, label=model_id.split("/")[-1], params=got["params"],
            revision=harness.repo_revision(model_id), dtype=str(next(mdl.parameters()).dtype),
            tokenizer_sha256=harness.tokenizer_fingerprint(tok),
            role="primary" if model_id == PRIMARY else "second curve", quant=quant)
        return mdl, tok

    print(f"generating on {GPU_NAME}")

## 5. Generating with a token budget

`run_at_budget` writes up to `budget` tokens. A chain that ends its turn first is **finished**; one
that reaches the limit is **cut off** (`truncated`) wherever it was. Each token is kept as a string
so the website can replay the chain.

In [ ]:
def prompt_ids_for(question, tok):
    """ch03's prompt in the model's chat template, as token ids. Both arms start here."""
    chat = tok.apply_chat_template([{"role": "user", "content": render_prompt(question)}],
                                   tokenize=False, add_generation_prompt=True)
    return tok(chat, add_special_tokens=False).input_ids


def run_at_budget(question, budget, mdl, tok, stop_ids):
    ids = harness.continue_ids(prompt_ids_for(question, tok), mdl, tok, budget)
    truncated = len(ids) >= budget              # fewer tokens means it stopped on its own
    while ids and ids[-1] in stop_ids:          # the end-of-turn token is not text
        ids.pop()
    tokens = harness.incremental_token_strings(ids, tok)
    return dict(trace="".join(tokens), tokens=tokens, truncated=truncated,
                own_tokens=len(ids), tokens_generated=len(ids))

## 6. Running every budget

Every model, every budget, smallest first. A chain that finished on its own at a smaller budget is
reused at the larger ones (`carried_from`), because greedy decoding would write it identically;
check 4 in section 8 confirms one.

In [ ]:
def reusable(model_id, budget, problem_id):
    """A chain this model already finished on its own at a smaller budget, if any."""
    for smaller in reversed([b for b in BUDGETS if b < budget]):
        prev = STORE.done.get((model_id, smaller, problem_id))
        if prev and not prev["truncated"]:
            return prev
    return None


def sweep(model_id, mdl, tok):
    stop_ids = harness.generation_stop_ids(mdl)
    results[model_id] = {}
    for b in BUDGETS:                      # ascending, so a reusable chain always exists already
        for row in harness.progress(items, desc=f"{model_id.split('/')[-1]} @ {b}", unit="problem"):
            if (model_id, b, row["unique_id"]) in STORE.done:
                continue                   # already on disk from an earlier, interrupted run
            prev = reusable(model_id, b, row["unique_id"])
            if prev is not None:
                STORE.append(dict(prev, budget=b, carried_from=prev.get("carried_from") or prev["budget"]))
                continue
            g = run_at_budget(row["problem"], b, mdl, tok, stop_ids)
            answer, correct = grade_chain(g["trace"], g["truncated"], row["answer"])
            STORE.append(dict(
                model_id=model_id, problem_id=row["unique_id"], level=int(row["level"]),
                subject=row["subject"], question=row["problem"], ground_truth=row["answer"],
                budget=b, extracted_answer=answer, correct=correct, carried_from=None, **g))
        results[model_id][b] = [STORE.done[(model_id, b, r["unique_id"])] for r in items]


def spot_check_reuse(model_id, mdl, tok):
    """Regenerate one reused record directly and confirm it is identical."""
    r = next((r for b in BUDGETS for r in results[model_id][b] if r.get("carried_from")), None)
    if r is None:
        return None
    again = run_at_budget(r["question"], r["budget"], mdl, tok, harness.generation_stop_ids(mdl))
    check = dict(problem_id=r["problem_id"], budget=r["budget"],
                 carried_from=r["carried_from"], identical=again["trace"] == r["trace"])
    assert check["identical"], "a reused record does not reproduce when generated directly"
    return check


if GENERATES:
    CARRY_CHECK = None
    for model_id, quant in MODELS:         # one model in memory at a time
        mdl, tok = load(model_id, quant)
        sweep(model_id, mdl, tok)
        if model_id == PRIMARY:
            CARRY_CHECK = spot_check_reuse(model_id, mdl, tok)
        del mdl
        harness.free_memory()
else:
    CARRY_CHECK = RUN_MANIFEST["integrity"]["carry_forward_spot_check"]

SWEPT = [model_id for model_id, _ in MODELS if model_id in results]
for model_id in SWEPT:
    for b in BUDGETS:
        results[model_id][b].sort(key=lambda r: ORDER[r["problem_id"]])
        assert len(results[model_id][b]) == len(items), f"{model_id} at {b} tokens is missing problems"

print(f"{'problems solved':<24}" + "".join(f"{b:>8}" for b in BUDGETS))
for model_id in SWEPT:
    print(f"{model_id.split('/')[-1]:<24}"
          + "".join(f"{sum(r['correct'] for r in results[model_id][b]):>8}" for b in BUDGETS))

## 7. One problem, every budget

What the records look like for a single problem: how much the model wrote, whether the budget cut
it off, and how it was graded.

In [ ]:
def show_trace(r, width=94, head=4, tail=2):
    """The model's own words, marking where the budget cut it off."""
    lines = textwrap.wrap(r["trace"].strip().replace("\n", " "), width) or ["(nothing generated)"]
    if len(lines) > head + tail:
        lines = lines[:head] + [f"... [{len(lines) - head - tail} lines] ..."] + lines[-tail:]
    out = [f"  {line}" for line in lines]
    if r["truncated"]:
        out[-1] += "  [out of tokens]"
    return "\n".join(out)


example = items[0]
print(textwrap.fill("Q: " + example["problem"], 96))
print("reference answer:", example["answer"])
for b in BUDGETS:
    r = next(r for r in results[PRIMARY][b] if r["problem_id"] == example["unique_id"])
    print(f"\n--- {b} tokens: wrote {r['own_tokens']}, {'cut off' if r['truncated'] else 'finished'}, "
          f"{'correct' if r['correct'] else 'wrong'}")
    print(show_trace(r))

## 8. Checks on the run

1. Every chain rebuilds exactly from its stored tokens.
2. No chain is longer than its budget.
3. A chain cut off at a smaller budget is the beginning of the same problem's longest chain: the
   budget only cuts, it never changes what the model writes.
4. A reused record is identical to one generated directly (section 6).
5. How often the budget cut chains off. At the top budget it should rarely happen.

In [ ]:
all_records = [r for model_id in SWEPT for b in BUDGETS for r in results[model_id][b]]
reused = [r for r in all_records if r.get("carried_from")]

assert all("".join(r["tokens"]) == r["trace"] for r in all_records), "1. a chain does not rebuild"
assert all(r["own_tokens"] <= r["budget"] for r in all_records), "2. a chain overran its budget"

# 3. cut chains are prefixes of the longest chain for the same problem
comparisons = []
for model_id in SWEPT:
    longest = {r["problem_id"]: r for r in results[model_id][MAX_BUDGET]}
    for b in BUDGETS[:-1]:
        for r in results[model_id][b]:
            if r["truncated"]:             # reused records would just be compared with themselves
                comparisons.append(harness.prefix_len(r["tokens"], longest[r["problem_id"]]["tokens"])
                                   == len(r["tokens"]))
prefix_exact = mean(comparisons)

INTEGRITY = dict(n_records=len(all_records), n_cut_off=sum(r["truncated"] for r in all_records),
                 prefix_exact_rate=prefix_exact, n_prefix_comparisons=len(comparisons),
                 n_carried_forward=len(reused), carry_forward_spot_check=CARRY_CHECK)

print(f"1-2. {len(all_records):,} chains rebuild from their tokens and stay within budget")
print(f"3.   cut chains that are exact prefixes of the longer chain: {prefix_exact:.1%} of {len(comparisons)}")
print(f"4.   reused records: {len(reused)}; spot check: "
      + ("none needed" if CARRY_CHECK is None else
         f"{CARRY_CHECK['problem_id']} at {CARRY_CHECK['budget']} -> "
         f"{'identical' if CARRY_CHECK['identical'] else 'DIFFERENT'}"))
print("5.   share of chains cut off by the budget:")
print(f"     {'budget':>8}" + "".join(f"{m.split('/')[-1]:>24}" for m in SWEPT))
for b in BUDGETS:
    print(f"     {b:>8}" + "".join(f"{mean([r['truncated'] for r in results[m][b]]):>24.0%}" for m in SWEPT))

top = results[PRIMARY][MAX_BUDGET][:min(30, len(items))]
top_cut = mean([r["truncated"] for r in top])
PILOT = dict(n=len(top), budget=MAX_BUDGET, accuracy=mean([r["correct"] for r in top]),
             mean_own_tokens=mean([r["own_tokens"] for r in top]), truncation_rate=top_cut,
             verdict="grid brackets the model" if top_cut <= 0.10 else "TOP OF GRID STILL BINDS",
             advice="proceed" if top_cut <= 0.10 else "add the next doubling to BUDGETS")
print(f"\ntop budget ({MAX_BUDGET}): {top_cut:.0%} of the first {len(top)} chains still cut off -> {PILOT['verdict']}")

if RUN_MODE == "REPLAY":
    keys = ("n_records", "prefix_exact_rate", "n_prefix_comparisons", "n_carried_forward")
    assert all(INTEGRITY[k] == RUN_MANIFEST["integrity"][k] for k in keys), \
        "the checks differ from the ones recorded by the run that generated these chains"
    print("the checks match the ones recorded by the run that generated these chains")

# Results

## 9. Figure 1: every problem, budget and model

One cell per problem, budget and model. A cell is **solved** (cyan), **answered but wrong**
(magenta), or **cut off** by the budget before it answered (pale, with a scissor mark). The number
in a coloured cell is how many tokens the model wrote. Rows are ordered by the budget at which the
larger model first solves them.

Read across a row and the budget's effect on one problem is visible directly: cut off while the
budget is too small, then answered (right or wrong) once the chain has room to finish.

The drawing lives in `page_06_heatmap.py`, which `docs/review/page6_small_run/build_heatmap.py`
also uses. The figure is saved as `fig1_heatmap.svg` beside the run.

In [ ]:
import page_06_heatmap as heatmap
from IPython.display import SVG, display

HEATMAP = heatmap.from_records(results, SWEPT, BUDGETS, run_id=RUN_ID)
FIG1_PATH = OUT_DIR / "fig1_heatmap.svg"
FIG1_PATH.write_text(heatmap.svg(HEATMAP), encoding="utf-8")
display(SVG(filename=str(FIG1_PATH)))

# Say so when the top budget changes nothing: every chain had already finished one step below.
top, below = BUDGETS[-1], BUDGETS[-2]
same = all(heatmap.state(p["cells"][(m, top)]) == heatmap.state(p["cells"][(m, below)])
           for p in HEATMAP["problems"] for m in HEATMAP["models"])
print(f"{len(HEATMAP['problems'])} problems x {len(BUDGETS)} budgets x {len(SWEPT)} models -> {FIG1_PATH}")
if same:
    print(f"the {top:,} budget gives the same outcome as {below:,} on every problem")

## 10. Figure 2: accuracy against the token budget

Share of the 100 problems each model solves at each budget. The table under the plot reports what
each doubling of the budget bought, and the budget past which no later doubling buys two points or
more (where the curve "stops paying"). A dip is kept, not smoothed away.

In [ ]:
def curve_for(model_id):
    pts = []
    for b in BUDGETS:
        rs = results[model_id][b]
        pts.append(dict(budget=b, n=len(rs),
                        n_correct=sum(r["correct"] for r in rs), accuracy=mean([r["correct"] for r in rs]),
                        mean_tokens=mean([r["tokens_generated"] for r in rs]),
                        total_tokens=sum(r["tokens_generated"] for r in rs),
                        truncation_rate=mean([r["truncated"] for r in rs])))
    return pts


def bend(pts):
    """What each doubling bought, and where the curve stops paying (no later gain of 2+ points)."""
    gains = [dict(from_budget=a["budget"], to_budget=b["budget"], gain_pts=(b["accuracy"] - a["accuracy"]) * 100)
             for a, b in zip(pts, pts[1:])]
    knee = next((g["from_budget"] for i, g in enumerate(gains)
                 if all(h["gain_pts"] < 2.0 for h in gains[i:])), None)
    top_ = max(p["accuracy"] for p in pts)
    return dict(gains=gains, monotone=all(g["gain_pts"] >= 0 for g in gains), knee_budget=knee,
                ceiling=top_, floor=pts[0]["accuracy"], total_range_pts=(top_ - pts[0]["accuracy"]) * 100)


CURVES = {m: curve_for(m) for m in SWEPT}
BEND = {m: bend(CURVES[m]) for m in SWEPT}

fig, ax = plt.subplots(figsize=(8, 4.2))
for (model_id, pts), colour in zip(CURVES.items(), ["#3b4a6b", "#a2704a", "#5c7a5c"]):
    ax.plot(BUDGETS, [p["accuracy"] * 100 for p in pts], marker="o", ms=4, lw=2, color=colour,
            label=model_id.split("/")[-1])
ax.set_xscale("log", base=2)
ax.set_xticks(BUDGETS)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("token budget (each step doubles it)")
ax.set_ylabel("problems solved (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_2_curve.png", dpi=160)
plt.show()

for model_id in SWEPT:
    print(f"\n{model_id.split('/')[-1]}   gain per doubling")
    for g in BEND[model_id]["gains"]:
        print(f"  {g['from_budget']:>5} -> {g['to_budget']:<5} {g['gain_pts']:>+6.1f} points")
    print(f"  stops paying after: {BEND[model_id]['knee_budget'] or 'still rising at the top budget'}")

## 11. Figure 3: when each problem first comes right

One row per problem, filled from the budget at which it is first graded correct. A problem that is
right at one budget and wrong at a larger one (a regression) is counted, not hidden.

In [ ]:
def by_problem(model_id):
    """problem_id -> {budget: record}, in problem order."""
    out = {}
    for b in BUDGETS:
        for r in results[model_id][b]:
            out.setdefault(r["problem_id"], {})[b] = r
    return out


def first_correct(row):
    return next((b for b in BUDGETS if row[b]["correct"]), None)


PRIMARY_BY_PROBLEM = by_problem(PRIMARY)

flip_items = []
for pid, row in PRIMARY_BY_PROBLEM.items():
    verdicts = [bool(row[b]["correct"]) for b in BUDGETS]
    first = first_correct(row)
    top_rec = row[MAX_BUDGET]
    flip_items.append(dict(problem_id=pid, level=top_rec["level"], subject=top_rec["subject"],
                           correct_by_budget=verdicts, first_correct_budget=first,
                           monotone=all(verdicts[BUDGETS.index(first):]) if first else True,
                           tokens_at_max=top_rec["tokens_generated"], truncated_at_max=top_rec["truncated"]))

# never-solved problems last, then by the budget they turn correct at, then by chain length
flip_items.sort(key=lambda d: (d["first_correct_budget"] is None,
                               BUDGETS.index(d["first_correct_budget"]) if d["first_correct_budget"] else 0,
                               d["tokens_at_max"]))
never = [d for d in flip_items if d["first_correct_budget"] is None]
STRIP = dict(histogram={str(b): sum(d["first_correct_budget"] == b for d in flip_items) for b in BUDGETS},
             never_flip_count=len(never), never_flip_share=len(never) / len(flip_items),
             n_regressions=sum(not d["monotone"] for d in flip_items))

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(np.array([d["correct_by_budget"] for d in flip_items], dtype=int), aspect="auto",
          interpolation="nearest", cmap=ListedColormap(["#e8e6e1", "#3b4a6b"]), vmin=0, vmax=1)
ax.set_xticks(range(len(BUDGETS)))
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_yticks([])
ax.set_xlabel("token budget")
ax.set_ylabel(f"problems (n={len(flip_items)}), sorted by when they first come right")
ax.set_title(PRIMARY.split("/")[-1], fontsize=9, loc="left", color="#555")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_3_flip_strip.png", dpi=160)
plt.show()

for b in BUDGETS:
    print(f"first correct at {b:>5} tokens: {STRIP['histogram'][str(b)]:>3} problems")
print(f"never correct:              {STRIP['never_flip_count']:>3} problems")
print(f"regressions (right, then wrong at a larger budget): {STRIP['n_regressions']}")

## 12. The opposite of a cap: telling the model to keep going

When the model tries to end its turn, the end-of-turn token is dropped, `Wait` is appended and it
carries on, up to three times, graded after each (budget forcing, Muennighoff et al., 2025). These
models have no thinking section and no reasoning fine-tune, so the word lands after a finished answer.

- `k = 0` is the model left alone at the top budget, and must match the main sweep.
- Each `Wait` buys at most 512 tokens; a chain still going then is cut off and graded like any other.
- Fixed and broken counts are reported over all problems, and over the clean ones where nothing was
  cut off.

In [ ]:
WAIT_TEXT = "\n" + WAIT_WORD


def run_with_waits(question, mdl, tok, stop_ids, n_waits):
    """One growing chain, sampled after each "Wait": record k is record k-1 plus our word
    plus whatever the model wrote next."""
    prompt_ids = prompt_ids_for(question, tok)
    wait_ids = tok(WAIT_TEXT, add_special_tokens=False).input_ids

    body, spans, out = [], [], []
    for k in range(n_waits + 1):
        room = MAX_BUDGET if k == 0 else WAIT_ALLOWANCE
        fresh = harness.continue_ids(prompt_ids + body, mdl, tok, room)
        cut_off = len(fresh) >= room                   # it never offered to stop
        while fresh and fresh[-1] in stop_ids:
            fresh.pop()
        body += fresh
        tokens = harness.incremental_token_strings(body, tok)
        own = len(body) - sum(e - s for s, e in spans)  # our injected words are not its cost
        out.append(dict(trace="".join(tokens), tokens=tokens, truncated=cut_off, own_tokens=own,
                        tokens_generated=own, waits=k, wait_spans=[list(s) for s in spans],
                        offered_a_stop=not cut_off))
        if cut_off:
            break                                       # nothing left to suppress
        spans.append((len(body), len(body) + len(wait_ids)))
        body += wait_ids
    return out


if GENERATES and RUN_WAIT_ARM:
    # Resume from the wait records already on disk. They share traces.jsonl with the
    # budget sweep but are keyed by the number of waits, so they are read here directly.
    done = {}
    with TRACES_PATH.open(encoding="utf-8") as f:
        for r in (json.loads(line) for line in f if line.strip()):
            if r.get("run_id") == RUN_ID and r.get("arm") == "wait":
                done[(r["model_id"], r["waits"], r["problem_id"])] = r
    for model_id, quant in MODELS:
        mdl, tok = load(model_id, quant)
        stop_ids = harness.generation_stop_ids(mdl)
        for row in harness.progress(items, desc=f"waits {model_id.split('/')[-1]}", unit="problem"):
            if all((model_id, k, row["unique_id"]) in done for k in WAITS):
                continue
            snaps = {s["waits"]: s for s in run_with_waits(row["problem"], mdl, tok, stop_ids, max(WAITS))}
            last = None
            for k in WAITS:
                # A chain cut off before step k has nothing left to inject into, so step k is
                # the last real record carried forward, relabelled with the step it stands for.
                s, carried = (snaps[k], None) if k in snaps else (dict(last, waits=k), last["waits"])
                answer, correct = grade_chain(s["trace"], s["truncated"], row["answer"])
                done[(model_id, k, row["unique_id"])] = STORE.append(dict(
                    model_id=model_id, problem_id=row["unique_id"], level=int(row["level"]),
                    subject=row["subject"], question=row["problem"], ground_truth=row["answer"],
                    arm="wait", budget=None, wait_word=WAIT_WORD, extracted_answer=answer,
                    correct=correct, carried_from=carried, **s))
                if k in snaps:
                    last = s
        wait_records[model_id] = [done[(model_id, k, r["unique_id"])] for r in items for k in WAITS]
        del mdl
        harness.free_memory()

for model_id in wait_records:
    wait_records[model_id].sort(key=lambda r: (ORDER[r["problem_id"]], r["waits"]))

# WAIT_BY[model][problem][k]: the checks and the specimens below read this.
WAIT_BY = {m: {} for m in wait_records}
for model_id, rs in wait_records.items():
    for r in rs:
        WAIT_BY[model_id].setdefault(r["problem_id"], {})[r["waits"]] = r
WAIT_LEAD = PRIMARY if PRIMARY in wait_records else min(wait_records, default=None)
print(sum(len(v) for v in wait_records.values()), "wait-arm records")

In [ ]:
WAIT_CHECKS = {}
if wait_records:
    flat = [r for rs in wait_records.values() for r in rs]

    # k = 0 is the model left alone: it must match the main sweep at the top budget.
    finished = {r["problem_id"]: r for r in results[WAIT_LEAD][MAX_BUDGET] if not r["truncated"]}
    zero = [r for r in wait_records[WAIT_LEAD] if r["waits"] == 0 and r["problem_id"] in finished]
    same = sum(r["trace"] == finished[r["problem_id"]]["trace"] for r in zero)
    WAIT_CHECKS["k0_matches_uncapped"] = (same, len(zero))

    assert all("".join(r["tokens"]) == r["trace"] for r in flat), "a chain does not rebuild"
    breaks = sum(not ks[b]["trace"].startswith(ks[a]["trace"])  # each k extends the one before
                 for by in WAIT_BY.values() for ks in by.values()
                 for a, b in zip(WAITS, WAITS[1:]) if a in ks and b in ks)
    WAIT_CHECKS["prefix_breaks"] = breaks
    assert not breaks, "a wait-arm record does not extend the one before it"
    miscounted = [r for r in flat if r["own_tokens"] != len(r["tokens"]) - sum(e - s for s, e in r["wait_spans"])]
    WAIT_CHECKS["miscounted"] = len(miscounted)
    assert not miscounted, "an injected word was counted as the model's own output"
    WAIT_CHECKS["cut_off_at_kmax"] = {
        m: mean([r["truncated"] for r in rs if r["waits"] == max(WAITS)]) for m, rs in wait_records.items()}

    print(f"k = 0 matches the main sweep: {same} of {len(zero)} chains identical")
    print("every record rebuilds, extends the one before it, and excludes our words from its token count")
    print(f"cut off by the {WAIT_ALLOWANCE}-token allowance at the last step: "
          + ", ".join(f"{m.split('/')[-1]} {v:.0%}" for m, v in WAIT_CHECKS["cut_off_at_kmax"].items()))

In [ ]:
def same_answer(a, b):
    """Equivalent answers count as the same answer ("4" and "4.0")."""
    return a == b or (bool(a) and bool(b) and bool(grade_answer(a, b)))


WAIT_CURVES, WAIT_PAIRED, WAIT_VERDICT = {}, {}, {}
for model_id, rs in wait_records.items():
    by_k = {}
    for r in rs:
        by_k.setdefault(r["waits"], {})[r["problem_id"]] = r
    base = by_k[WAITS[0]]
    WAIT_CURVES[model_id] = []
    for k in WAITS:
        at_k = [r for r in rs if r["waits"] == k]
        moved = [not same_answer(r["extracted_answer"], base[r["problem_id"]]["extracted_answer"]) for r in at_k]
        WAIT_CURVES[model_id].append(dict(
            waits=k, n=len(at_k), churn=mean(moved) if moved else 0.0,
            n_correct=sum(r["correct"] for r in at_k), accuracy=mean([r["correct"] for r in at_k]),
            mean_tokens=mean([r["own_tokens"] for r in at_k]), total_tokens=sum(r["own_tokens"] for r in at_k),
            offered_a_stop=mean([r["offered_a_stop"] for r in at_k]),
            carried=mean([r["carried_from"] is not None for r in at_k])))

    # Problems fixed and broken against k = 0, with an exact McNemar test (as on page 5).
    WAIT_PAIRED[model_id] = []
    for k in WAITS[1:]:
        pids = sorted(set(base) & set(by_k[k]))
        fixed = [p for p in pids if by_k[k][p]["correct"] and not base[p]["correct"]]
        broken = [p for p in pids if base[p]["correct"] and not by_k[k][p]["correct"]]
        _, _, p_value = harness.mcnemar({q: by_k[k][q]["correct"] for q in pids}, {q: base[q]["correct"] for q in pids})
        clean = [p for p in pids if not by_k[k][p]["truncated"] and not base[p]["truncated"]
                 and by_k[k][p].get("carried_from") is None]
        c_fixed, c_broken, c_p = harness.mcnemar({q: by_k[k][q]["correct"] for q in clean},
                                                 {q: base[q]["correct"] for q in clean})
        WAIT_PAIRED[model_id].append(dict(waits=k, fixed=len(fixed), broken=len(broken), p=p_value, n=len(pids),
                                          fixed_ids=fixed, broken_ids=broken,
                                          clean=dict(n=len(clean), fixed=c_fixed, broken=c_broken, p=c_p,
                                                     excluded=len(pids) - len(clean))))

    pts, top_ = WAIT_CURVES[model_id], WAIT_PAIRED[model_id][-1]
    delta, churn = (pts[-1]["accuracy"] - pts[0]["accuracy"]) * 100, pts[-1]["churn"]
    verdict = ("the injections help" if delta >= 2 and top_["p"] < 0.05 else
               "the injections do nothing: it restates the answer it already had" if churn < 0.05 else
               "the injections hurt" if delta <= -2 and top_["p"] < 0.05 else
               "the injections churn without gaining")
    WAIT_VERDICT[model_id] = dict(delta_pts=delta, churn=churn, fixed=top_["fixed"], broken=top_["broken"],
                                  p=top_["p"], verdict=verdict)
    print(f"{model_id.split('/')[-1]:<24}{pts[0]['accuracy']:.0%} -> {pts[-1]['accuracy']:.0%} after "
          f"{WAITS[-1]} waits; {churn:.0%} of answers moved; {top_['fixed']} fixed, {top_['broken']} broken "
          f"(p = {top_['p']:.3f}): {verdict}")

if WAIT_CURVES:
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for (model_id, pts), colour in zip(WAIT_CURVES.items(), ["#3b4a6b", "#a2704a", "#5c7a5c"]):
        ax.plot(WAITS, [p["accuracy"] * 100 for p in pts], marker="o", ms=4, lw=2, color=colour,
                label=model_id.split("/")[-1])
    ax.set_xticks(WAITS)
    ax.set_xlabel(f'times "{WAIT_WORD}" was appended')
    ax.set_ylabel("problems solved (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_4_waits.png", dpi=160)
    plt.show()

## 13. Save the results

`results.json` holds every number above; `manifest.json` records what was run. In `REPLAY` the
manifest names this notebook and keeps the generating run's record under `generated_by`.

In [ ]:
AT_THE_BUDGET = "generation stops; a chain cut off before writing a boxed answer is graded wrong"

if GENERATES:
    generated_by = None
    run_record = dict(
        models=[model_meta[m] for m in SWEPT], dtype_requested=str(DTYPE),
        decoding={"strategy": "greedy", "do_sample": False, "temperature": None, "top_p": None, "top_k": None},
        tf32_matmul=bool(torch.backends.cuda.matmul.allow_tf32),
        budgets=BUDGETS,
        waits={"word": WAIT_WORD, "grid": WAITS, "allowance": WAIT_ALLOWANCE, "injection": WAIT_TEXT,
               "where": "appended when the model tries to end its turn; these models have no "
                        "thinking section, so that is after a finished answer"},
        items={"source": "MATH-500 (ch03 copy)", "dataset": "math500_l45", "n": len(items),
               "item_set_sha256": ITEM_SET_HASH, "seed": SEED},
        grading={"extract": "reasoning_from_scratch.ch03.extract_final_candidate",
                 "extract_when_cut_off": "reasoning_from_scratch.ch03.get_last_boxed",
                 "grade": "reasoning_from_scratch.ch03.grade_answer"},
        prompt_template=render_prompt("{question}"),
        chat_template=tok.apply_chat_template([{"role": "user", "content": render_prompt("What is 2+2?")}],
                                              tokenize=False, add_generation_prompt=True),
        versions={"torch": torch.__version__, "transformers": __import__("transformers").__version__,
                  "device": str(device), "gpu": GPU_NAME},
    )
else:
    # The run that generated the chains keeps its own record beside this notebook's.
    REPLACED = ("at_the_budget", "integrity")
    generated_by = RUN_MANIFEST.get("generated_by") or {
        k: RUN_MANIFEST[k] for k in harness.RUN_PROVENANCE + REPLACED if k in RUN_MANIFEST}
    run_record = {k: v for k, v in RUN_MANIFEST.items() if k not in harness.MANIFEST_OWN + REPLACED}

harness.write_manifest(MANIFEST_PATH, page=6, notebook="page_06_thinking_longer.ipynb", run_id=RUN_ID,
                       run_mode=RUN_MODE, generated_by=generated_by, at_the_budget=AT_THE_BUDGET,
                       integrity=INTEGRITY, **run_record)

harness.write_json(RESULTS_PATH, dict(
    run_id=RUN_ID, primary_model=PRIMARY, models=SWEPT, problem_ids=list(ORDER),
    budgets=BUDGETS, waits=WAITS, pilot=PILOT,
    curves={m: dict(points=CURVES[m], **BEND[m]) for m in SWEPT},
    flip_strip=dict(model_id=PRIMARY, items=flip_items, **STRIP),
    wait=None if not WAIT_CURVES else dict(
        lead_model=WAIT_LEAD, word=WAIT_WORD, allowance=WAIT_ALLOWANCE, curves=WAIT_CURVES,
        paired=WAIT_PAIRED, verdicts=WAIT_VERDICT, checks=WAIT_CHECKS),
))
for path in (RESULTS_PATH, MANIFEST_PATH, TRACES_PATH):
    print(f"{path.name:<15}{harness.kb(path):>9,.0f} KB")